In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader

sentences = ["what is statquest <EOS> awesome",
             "statquest is what <EOS> awesome",
             "squatch eats pizza <EOS> yum",    # the usage of position encoding
             "pizza eats squatch <EOS> yikes"]

token_to_id = {'what': 0,
               'is': 1,
               'statquest': 2,
               'awesome': 3,
               "squatch": 4,
               "eats": 5,
               "pizza": 6,
               "yum": 7,
               "yikes": 8,
               '<EOS>': 9} # <EOS> = end of sequence

id_to_token = dict(map(reversed, token_to_id.items()))

inputs = torch.tensor([[token_to_id[token] for token in sentence.split()]
                       for sentence in sentences])
# what is statquest <EOS> awesome
# statquest is what <EOS> awesome

labels = torch.tensor([[token_to_id[token] for token in sentence.split()[1:]] + [token_to_id['<EOS>']]
                       for sentence in sentences])
# is statquest <EOS> awesome <EOS>
# is what <EOS> awesome <EOS>

dataset = TensorDataset(inputs, labels)
dataloader = DataLoader(dataset)

In [2]:
from importlib import reload
from src import transformer
reload(transformer)
from src.transformer import DecoderOnlyTransformer

import lightning as L

max_length = 6
model = DecoderOnlyTransformer(num_tokens=len(token_to_id), d_model=2, max_len=max_length)

In [5]:
### Check

eos_id = token_to_id['<EOS>']

model_input = ["what is statquest <EOS>",
               "statquest is what <EOS>",
               "squatch eats pizza <EOS>",
               "pizza eats squatch <EOS>"]

model_input = torch.tensor([[token_to_id[token] for token in sentence.split()]
                           for sentence in model_input])
input_length = model_input.size(dim=1)
predicted_ids = torch.tensor([])
for _ in range(input_length, max_length):
  for batch in model_input:
    print(f"Model Input:\n{' '.join([id_to_token[id.item()] for id in batch])}")
  predictions = model(model_input)
  predicted_id = torch.argmax(predictions[:, -1:], dim=-1)
  # predicted_id: [batch_size, 1]
  predicted_ids = torch.cat([predicted_ids, predicted_id], dim=-1)

  model_input = torch.cat([model_input, predicted_id], dim=-1)

print("Predicted Tokens:")
for batch in predicted_ids:
  print(" ".join([id_to_token[id.item()] for id in batch]))

Model Input:
what is statquest <EOS>
Model Input:
statquest is what <EOS>
Model Input:
squatch eats pizza <EOS>
Model Input:
pizza eats squatch <EOS>
Model Input:
what is statquest <EOS> awesome
Model Input:
statquest is what <EOS> awesome
Model Input:
squatch eats pizza <EOS> yum
Model Input:
pizza eats squatch <EOS> yikes
Predicted Tokens:
awesome <EOS>
awesome <EOS>
yum <EOS>
yikes <EOS>


In [4]:
### Training

trainer = L.Trainer(max_epochs=30)
trainer.fit(model, train_dataloaders=dataloader)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/jason/.pyenv/versions/3.10.14/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default

  | Name           | Type             | Params | Mode 
------------------------------------------------------------
0 | we             | Embedding        | 20     | train
1 | pe             | PositionEncoding | 0      | train
2 | self_attention | Attention        | 12     | train
3 | fc_layer       | Linear           | 30     | trai

Training: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.
